In [3]:
import tkinter as tk
import copy

board = [[2,8,3],
         [1,6,4],
         [7,0,5]]

goal = [[1,2,3],
        [8,0,4],
        [7,6,5]]

move = {
    (-1,0):"UP",
    (1,0):"DOWN",
    (0,-1):"LEFT",
    (0,1):"RIGHT"
}


def goal_test(state):
    return state == goal


def find_zero(state):

    for i in range(3):
        for j in range(3):
            if state[i][j]==0:
                return i,j


class Node:

    def __init__(
            self,
            state,
            parent,
            action,
            cost):

        self.state=state
        self.parent=parent
        self.action=action
        self.cost=cost


def in_path(cur_node,state):

    while cur_node:

        if cur_node.state==state:
            return True

        cur_node=cur_node.parent

    return False


# Generator cho trực quan hóa
def DLS(problem,limit):

    stack=[]

    root=Node(
        copy.deepcopy(problem),
        None,
        None,
        0
    )

    stack.append(root)

    direction=[
        (0,1),
        (0,-1),
        (1,0),
        (-1,0)
    ]

    result="failure"

    while stack:

        cur=stack.pop()

        yield (
            cur.state,
            cur.cost,
            cur.action
        )

        if goal_test(cur.state):
            return "goal"

        if cur.cost>=limit:

            result="cutoff"

        else:

            x,y=find_zero(
                cur.state
            )

            for dx,dy in direction:

                nx=x+dx
                ny=y+dy

                if(
                    0<=nx<3 and
                    0<=ny<3
                ):

                    new_state=copy.deepcopy(
                        cur.state
                    )

                    new_state[x][y],new_state[nx][ny]=\
                    new_state[nx][ny],new_state[x][y]

                    if not in_path(
                        cur,
                        new_state
                    ):

                        child=Node(
                            new_state,
                            cur,
                            move[(dx,dy)],
                            cur.cost+1
                        )

                        stack.append(child)

    return result


class PuzzleGUI:

    def __init__(self,root):

        self.root=root

        root.title(
            "Iterative Deepening Search"
        )

        self.cells=[]

        frame=tk.Frame(root)
        frame.pack(
            pady=15
        )

        for i in range(3):

            row=[]

            for j in range(3):

                lbl=tk.Label(
                    frame,
                    width=4,
                    height=2,
                    font=(
                        "Arial",
                        22
                    ),
                    relief="solid"
                )

                lbl.grid(
                    row=i,
                    column=j,
                    padx=3,
                    pady=3
                )

                row.append(lbl)

            self.cells.append(
                row
            )

        self.info=tk.Label(
            root,
            font=("Arial",13)
        )

        self.info.pack()

        self.button=tk.Button(
            root,
            text="Start IDS",
            command=self.start
        )

        self.button.pack(
            pady=10
        )

        self.limit=0
        self.running=False


    def draw(self,state):

        for i in range(3):
            for j in range(3):

                value=state[i][j]

                self.cells[i][j]["text"]=(
                    ""
                    if value==0
                    else str(value)
                )


    def start(self):

        if self.running:
            return

        self.running=True
        self.button["state"]="disabled"

        self.limit=0

        self.run_limit()


    def run_limit(self):

        self.generator=DLS(
            board,
            self.limit
        )

        self.animate()


    def animate(self):

        try:

            state,cost,action=next(
                self.generator
            )

            self.draw(
                state
            )

            self.info["text"]=(
                f"Depth limit: {self.limit}"
                f"\nCurrent cost: {cost}"
                f"\nAction: {action}"
            )

            self.root.after(
                600,
                self.animate
            )

        except StopIteration as e:

            result=e.value

            if result=="goal":

                self.info["text"]+=\
                "\nGoal Found!"

                self.running=False
                return


            self.limit+=1

            self.root.after(
                1000,
                self.run_limit
            )


root=tk.Tk()

PuzzleGUI(root)

root.mainloop()